# Deepfake Detection System

This notebook compares a SmallCNN trained from scratch with pretrained ResNet18, VGG16, and Vision Transformer (ViT) models for binary real/fake image classification.

**Dataset:** [DeepFake vs Real 60K on Kaggle](https://www.kaggle.com/datasets/prithivsakthiur/deepfake-vs-real-60k)

Run the notebook from top to bottom. Training downloads pretrained weights, can take hours, and is best run with a supported GPU. The notebook uses a fixed random seed and reports validation metrics; inspect the dataset split warning before treating those metrics as a benchmark.

## 1. Setup

Use Python 3.11 or newer. Install a PyTorch and TorchVision pair that matches your operating system and compute platform using the [official PyTorch selector](https://pytorch.org/get-started/locally/), then install the project dependencies with:

```bash
python -m pip install -r requirements.txt
```

The dataset is downloaded with KaggleHub. If Kaggle requests account access or acceptance of dataset terms, complete that setup through Kaggle's supported flow; do not place credentials in notebook cells. A CPU can run the notebook, but training all four models may be very slow.

In [ ]:
import os, time, random
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import datasets, transforms, models
import timm

from tqdm import tqdm

from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score, roc_auc_score,
    confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, precision_recall_curve, average_precision_score
)
from sklearn.calibration import calibration_curve

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

### Reproducibility and runtime configuration

The configuration cell sets the random seed, split proportions, model hyperparameters, batch limit, and output directory. The default cap is 300 batches per epoch to make comparisons more practical; set max_batches_per_epoch to None to process each full epoch. GPU and library differences can still produce small numerical variation.

In [ ]:
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)

CFG = {
    "seed": 42,
    "img_size": 224,
    "batch_size": 32,
    "num_workers": min(4, os.cpu_count() or 1),
    "pin_memory": device == "cuda",
    "persistent_workers": True,
    "train_frac": 0.80,
    "val_frac": 0.10,
    "epochs_scratch": 4,
    "epochs_head": 2,
    "epochs_ft": 2,
    # Cap each epoch to keep the comparison practical; set to None for full epochs.
    "max_batches_per_epoch": 300,
    "lr_scratch": 1e-3,
    "lr_head": 1e-3,
    "lr_ft": 1e-4,
    "weight_decay": 1e-4,
    "use_amp": True,
    "save_dir": "artifacts/checkpoints",
}

seed_everything(CFG["seed"])
os.makedirs(CFG["save_dir"], exist_ok=True)


## 2. Dataset

KaggleHub downloads the public dataset identified above. The notebook detects the ImageFolder root (a directory with one subdirectory per class). Review the class names and image count before training. The code does not redistribute dataset images.

In [ ]:
import kagglehub

DATASET_HANDLE = "prithivsakthiur/deepfake-vs-real-60k"
DATA_ROOT = Path(kagglehub.dataset_download(DATASET_HANDLE))
print("Dataset downloaded to:", DATA_ROOT)


In [ ]:
def find_imagefolder_root(root: Path):
    img_ext = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

    def looks_like_imagefolder(p: Path):
        if not p.is_dir(): return False
        subdirs = [x for x in p.iterdir() if x.is_dir()]
        if len(subdirs) < 2: return False
        # must contain at least one image in at least one subdir
        for sd in subdirs:
            for f in sd.rglob("*"):
                if f.suffix.lower() in img_ext:
                    return True
        return False

    if looks_like_imagefolder(root):
        return root

    candidates = []
    for p in root.rglob("*"):
        if p.is_dir() and looks_like_imagefolder(p):
            candidates.append(p)

    if not candidates:
        return None

    candidates.sort(key=lambda p: len(p.parts))
    return candidates[0]

imagefolder_root = find_imagefolder_root(DATA_ROOT)
print("Detected imagefolder_root:", imagefolder_root)

if imagefolder_root is None:
    raise RuntimeError(
        f"Could not detect ImageFolder root under: {DATA_ROOT}\n"
        f"Inspect the downloaded directory tree and point DATA_ROOT at the folder containing the class directories."
    )

### Image transforms and dataset wrapper

Training applies random horizontal flips; validation and test transforms are deterministic. The safe dataset wrapper retries a nearby sample when an image cannot be decoded. Check the displayed sample images before training.

In [ ]:
import torch
from torchvision import datasets, transforms

img_size = CFG["img_size"]

train_tfms = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])


eval_tfms = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])


raw_ds  = datasets.ImageFolder(str(imagefolder_root), transform=None)
full_ds = datasets.ImageFolder(str(imagefolder_root), transform=train_tfms)

print("Classes:", raw_ds.classes)
print("class_to_idx:", raw_ds.class_to_idx)
print("Total images:", len(raw_ds))

In [ ]:
class SafeImageFolder(Dataset):

    def __init__(self, root, transform=None):
        self.ds = datasets.ImageFolder(str(root), transform=transform)
        self.classes = self.ds.classes
        self.class_to_idx = self.ds.class_to_idx

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        try:
            return self.ds[idx]
        except Exception:
            # try a few next indices; if many corrupt, this still keeps moving
            for k in range(1, 10):
                j = (idx + k) % len(self.ds)
                try:
                    return self.ds[j]
                except Exception:
                    continue
            # if still failing, raise (extremely rare)
            raise

# For preview: raw dataset without transforms
raw_ds = SafeImageFolder(imagefolder_root, transform=None)
# For training: transformed dataset
full_ds = SafeImageFolder(imagefolder_root, transform=train_tfms)

print("Classes:", raw_ds.classes)
print("class_to_idx:", raw_ds.class_to_idx)
print("Total images:", len(raw_ds))

# Positive class: "fake" if present
class_to_idx = raw_ds.class_to_idx
idx_to_class = {v:k for k,v in class_to_idx.items()}
POS_CLASS_IDX = class_to_idx["fake"] if "fake" in class_to_idx else 1
print("Positive class used:", idx_to_class.get(POS_CLASS_IDX, POS_CLASS_IDX))

### Visual sanity check

In [ ]:
def show_original_samples(ds, n=3):
    plt.figure(figsize=(12,4))
    for i in range(n):
        img, label = ds[i]
        plt.subplot(1,n,i+1)
        plt.imshow(img)
        plt.title(f"Original | {ds.classes[label]}")
        plt.axis("off")
    plt.show()

def unnormalize(x, mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]):
    mean = torch.tensor(mean).view(3,1,1)
    std  = torch.tensor(std).view(3,1,1)
    return (x * std + mean).clamp(0,1)

def show_transformed_samples(ds, n=3):
    plt.figure(figsize=(12,4))
    for i in range(n):
        x, label = ds[i]
        x_vis = unnormalize(x.cpu()).permute(1,2,0).numpy()
        plt.subplot(1,n,i+1)
        plt.imshow(x_vis)
        plt.title(f"Transformed | {ds.ds.classes[label]}")
        plt.axis("off")
    plt.show()

show_original_samples(raw_ds, n=3)
show_transformed_samples(full_ds, n=3)

### Train, validation, and held-out test splits

The notebook creates an 80/10/10 split. The existing project notes possible dataset-integrity issues and therefore does not use the held-out test split for reported results. Validation metrics below are for model selection and should not be described as an independent test benchmark.

In [ ]:
n = len(full_ds)
n_train = int(CFG["train_frac"] * n)
n_val   = int(CFG["val_frac"] * n)
n_test  = n - n_train - n_val

train_ds, val_ds, test_ds = random_split(
    full_ds, [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(CFG["seed"])
)



val_base  = SafeImageFolder(imagefolder_root, transform=eval_tfms)
test_base = SafeImageFolder(imagefolder_root, transform=eval_tfms)


val_ds  = torch.utils.data.Subset(val_base,  val_ds.indices)
test_ds = torch.utils.data.Subset(test_base, test_ds.indices)


print("Split sizes:", len(train_ds), len(val_ds), len(test_ds))



### Data loaders

In [ ]:
batch_size = CFG["batch_size"]
num_workers = CFG["num_workers"]
persistent_workers = CFG["persistent_workers"] and num_workers > 0
loader_generator = torch.Generator().manual_seed(CFG["seed"])

train_loader = DataLoader(
    train_ds, batch_size=batch_size, shuffle=True,
    num_workers=num_workers, pin_memory=CFG["pin_memory"],
    persistent_workers=persistent_workers,
    worker_init_fn=seed_worker, generator=loader_generator
)
val_loader = DataLoader(
    val_ds, batch_size=batch_size, shuffle=False,
    num_workers=num_workers, pin_memory=CFG["pin_memory"],
    persistent_workers=persistent_workers
)
test_loader = DataLoader(
    test_ds, batch_size=batch_size, shuffle=False,
    num_workers=num_workers, pin_memory=CFG["pin_memory"],
    persistent_workers=persistent_workers
)
print("Loaders ready | batch_size:", batch_size, "| workers:", num_workers)


## 3. Training

The training cells define validation metrics and mixed-precision training, then fit the four models. Adjust the CFG values before running if the selected accelerator has limited memory.

### Evaluation helper used during training

In [ ]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    criterion = nn.CrossEntropyLoss()

    all_probs, all_preds, all_targets = [], [], []
    total_loss, total_seen = 0.0, 0

    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        logits = model(x)
        loss = criterion(logits, y)

        probs = torch.softmax(logits, dim=1)[:, POS_CLASS_IDX]
        preds = torch.argmax(logits, dim=1)

        bs = x.size(0)
        total_loss += loss.item() * bs
        total_seen += bs

        all_probs.append(probs.detach().cpu().numpy())
        all_preds.append(preds.detach().cpu().numpy())
        all_targets.append(y.detach().cpu().numpy())

    all_probs = np.concatenate(all_probs)
    all_preds = np.concatenate(all_preds)
    all_targets = np.concatenate(all_targets)

    # Binary labels for positive class
    y_bin = (all_targets == POS_CLASS_IDX).astype(int)
    preds_bin = (all_preds == POS_CLASS_IDX).astype(int)

    acc = accuracy_score(y_bin, preds_bin)
    f1  = f1_score(y_bin, preds_bin)
    prec = precision_score(y_bin, preds_bin, zero_division=0)
    rec  = recall_score(y_bin, preds_bin, zero_division=0)

    try:
        auc_val = roc_auc_score(y_bin, all_probs)
    except ValueError:
        auc_val = float("nan")

    cm = confusion_matrix(y_bin, preds_bin)
    avg_loss = total_loss / max(total_seen, 1)

    return {"loss": avg_loss, "acc": acc, "f1": f1, "precision": prec, "recall": rec, "auc": auc_val, "cm": cm}

@torch.no_grad()
def collect_preds(model, loader):
    model.eval()
    probs_list, y_list = [], []
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        logits = model(x)
        probs = torch.softmax(logits, dim=1)[:, POS_CLASS_IDX]
        probs_list.append(probs.cpu().numpy())
        y_list.append(y.numpy())
    probs = np.concatenate(probs_list)
    y = np.concatenate(y_list)
    y_bin = (y == POS_CLASS_IDX).astype(int)
    return probs, y_bin


### Model definitions

In [ ]:
num_classes = len(raw_ds.classes)

class SmallCNN(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2),  # 112

            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2),  # 56

            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d(2),  # 28

            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.MaxPool2d(2),  # 14
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x).flatten(1)
        return self.classifier(x)



In [ ]:
def build_resnet18(num_classes, pretrained=True):
    m = models.resnet18(weights=models.ResNet18_Weights.DEFAULT if pretrained else None)
    m.fc = nn.Linear(m.fc.in_features, num_classes)
    return m

In [ ]:
def build_vgg16(num_classes, pretrained=True):
    m = models.vgg16(weights=models.VGG16_Weights.DEFAULT if pretrained else None)
    m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, num_classes)
    return m

In [ ]:
def build_vit(num_classes, model_name="vit_small_patch16_224", pretrained=True):
    return timm.create_model(model_name, pretrained=pretrained, num_classes=num_classes)


### Training loop

In [ ]:
USE_AMP = CFG["use_amp"]

def train_one_epoch(model, loader, optimizer, scaler=None, max_batches=None):
    model.train()
    criterion = nn.CrossEntropyLoss()
    running_loss, seen = 0.0, 0

    for i, (x, y) in enumerate(loader, start=1):
        if max_batches is not None and i > max_batches:
            break

        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        if USE_AMP and device == "cuda":
            with torch.amp.autocast(device_type="cuda"):
                logits = model(x)
                loss = criterion(logits, y)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

        bs = x.size(0)
        running_loss += loss.item() * bs
        seen += bs

    return running_loss / max(seen, 1)

def fit(model, train_loader, val_loader, epochs, lr, tag="model"):
    model.to(device)
    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=CFG["weight_decay"]
    )
    scaler = torch.amp.GradScaler("cuda", enabled=(USE_AMP and device=="cuda"))

    best_auc = -1.0
    best_state = None

    for ep in range(1, epochs + 1):
        t0 = time.time()
        train_loss = train_one_epoch(
            model, train_loader, optimizer, scaler=scaler,
            max_batches=CFG["max_batches_per_epoch"]
        )
        val_m = evaluate(model, val_loader)
        dt = time.time() - t0

        print(f"[{tag}] Epoch {ep:02d}/{epochs} | {dt:6.1f}s | "
              f"train_loss={train_loss:.4f} | val_loss={val_m['loss']:.4f} "
              f"acc={val_m['acc']:.4f} f1={val_m['f1']:.4f} auc={val_m['auc']:.4f}")

        if not np.isnan(val_m["auc"]) and val_m["auc"] > best_auc:
            best_auc = val_m["auc"]
            best_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)

    return model

In [ ]:
E_S = CFG["epochs_scratch"]
E_H = CFG["epochs_head"]
E_F = CFG["epochs_ft"]

print("\n--- Training SmallCNN ---")
smallcnn = SmallCNN(num_classes=num_classes).to(device)
smallcnn = fit(smallcnn, train_loader, val_loader, epochs=E_S, lr=CFG["lr_scratch"], tag="SmallCNN")

print("\n--- Training ResNet18 (head then fine-tune) ---")
resnet18 = build_resnet18(num_classes=num_classes, pretrained=True).to(device)

# head-only
for p in resnet18.parameters(): p.requires_grad = False
for p in resnet18.fc.parameters(): p.requires_grad = True
resnet18 = fit(resnet18, train_loader, val_loader, epochs=E_H, lr=CFG["lr_head"], tag="ResNet18_head")

# fine-tune
for p in resnet18.parameters(): p.requires_grad = True
resnet18 = fit(resnet18, train_loader, val_loader, epochs=E_F, lr=CFG["lr_ft"], tag="ResNet18_ft")

print("\n--- Training VGG16 (head then fine-tune) ---")
vgg16 = build_vgg16(num_classes=num_classes, pretrained=True).to(device)

# head-only (classifier)
for p in vgg16.features.parameters(): p.requires_grad = False
for p in vgg16.classifier.parameters(): p.requires_grad = True
vgg16 = fit(vgg16, train_loader, val_loader, epochs=E_H, lr=CFG["lr_head"], tag="VGG16_head")

# fine-tune (heavier; can skip by setting epochs_ft=0)
if E_F > 0:
    for p in vgg16.parameters(): p.requires_grad = True
    vgg16 = fit(vgg16, train_loader, val_loader, epochs=E_F, lr=CFG["lr_ft"], tag="VGG16_ft")

print("\n--- Training ViT (head then fine-tune) ---")
vit = build_vit(num_classes=num_classes, model_name="vit_small_patch16_224", pretrained=True).to(device)

# head-only (timm usually has 'head')
for p in vit.parameters(): p.requires_grad = False
for name, p in vit.named_parameters():
    if "head" in name:
        p.requires_grad = True
vit = fit(vit, train_loader, val_loader, epochs=E_H, lr=CFG["lr_head"], tag="ViT_head")

# fine-tune
if E_F > 0:
    for p in vit.parameters(): p.requires_grad = True
    vit = fit(vit, train_loader, val_loader, epochs=E_F, lr=CFG["lr_ft"], tag="ViT_ft")

models_dict = {
    "SmallCNN": smallcnn,
    "ResNet18": resnet18,
    "VGG16": vgg16,
    "ViT": vit,
}


### Save trained weights and metadata

In [ ]:
import os, json, torch

SAVE_DIR = Path(CFG["save_dir"])
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# Save weights
for name, model in models_dict.items():
    torch.save(model.state_dict(), os.path.join(SAVE_DIR, f"{name}.pt"))

# Save metadata needed for inference/plots
meta = {
    "classes": getattr(raw_ds, "classes", None) or getattr(full_ds, "classes", None),
    "POS_CLASS_IDX": int(POS_CLASS_IDX),
    "img_size": int(CFG["img_size"]) if "CFG" in globals() and "img_size" in CFG else 224,
}
with open(os.path.join(SAVE_DIR, "meta.json"), "w") as f:
    json.dump(meta, f)

print("Saved models + meta to:", SAVE_DIR)
print("Files:", os.listdir(SAVE_DIR))


## 4. Evaluation

The final table and plot are computed from the trained models on the validation split. To claim final generalization performance, first resolve the dataset-integrity concern and evaluate the reserved test split once. Do not use the test split for repeated model tuning.

In [ ]:
# Validation metrics support model selection; they are not an independent test estimate.
results = []
for name, model in models_dict.items():
    metrics = evaluate(model, val_loader)
    results.append({
        "model": name,
        "val_loss": metrics["loss"],
        "val_accuracy": metrics["acc"],
        "val_precision": metrics["precision"],
        "val_recall": metrics["recall"],
        "val_f1": metrics["f1"],
        "val_roc_auc": metrics["auc"],
    })

validation_results = pd.DataFrame(results).sort_values("val_f1", ascending=False)
display(validation_results)

ax = validation_results.set_index("model")[["val_accuracy", "val_f1", "val_roc_auc"]].plot(
    kind="bar", figsize=(10, 5), ylim=(0, 1.05), ylabel="Score",
    title="Validation metrics by model"
)
ax.grid(True, axis="y", alpha=0.3)
